In [ ]:
from pathlib import Path

# Your data folder
DATA = Path(r"C:\Users\User\Desktop\Sanuth\data")
PROC = DATA / "processed"
print([p.name for p in DATA.iterdir()])

In [ ]:
import shutil, os
print("java:", shutil.which("java"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

import r5py
print("r5py version:", r5py.__version__)

In [ ]:
import zipfile

GTFS_DIR = DATA / "gtfs"
feeds = []
with zipfile.ZipFile(DATA / "gtfs.zip") as z:
    for mode in ["2", "3", "4"]:
        target = GTFS_DIR / mode / "google_transit.zip"
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(z.read(f"{mode}/google_transit.zip"))
        feeds.append(str(target))
feeds

In [ ]:
import pandas as pd
import geopandas as gpd

pts = pd.read_csv(PROC / "suburb_points.csv", dtype={"id": str})
origins = gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts.lon, pts.lat), crs=4326)

camp = pd.read_csv(PROC / "campuses.csv")
dests = gpd.GeoDataFrame(camp.rename(columns={"campus": "id"}),
                         geometry=gpd.points_from_xy(camp.lon, camp.lat), crs=4326)
print(len(origins), "suburbs,", len(dests), "campuses")

In [ ]:
network = r5py.TransportNetwork(str(DATA / "Melbourne.osm.pbf"), feeds)
print("network ready")

In [ ]:
import datetime

settings = dict(
    origins=origins,
    destinations=dests,
    departure=datetime.datetime(2026, 11, 17, 8, 0),
    departure_time_window=datetime.timedelta(hours=1),
    transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.WALK],
    max_time=datetime.timedelta(hours=2),
)

# r5py renamed this function between versions; this works with both
if hasattr(r5py, "TravelTimeMatrix"):
    tt = r5py.TravelTimeMatrix(network, **settings)
else:
    tt = r5py.TravelTimeMatrixComputer(network, **settings).compute_travel_times()

tt = pd.DataFrame(tt)
tt.to_csv(PROC / "travel_times.csv", index=False)
print(len(tt), "rows,", tt["travel_time"].isna().mean().round(3), "share empty")
tt.head()

In [ ]:
check = tt.merge(pts[["id", "suburb"]], left_on="from_id", right_on="id")
check[check.suburb.isin(["Footscray", "Clayton", "Box Hill", "Brunswick"])].pivot(index="suburb", columns="to_id", values="travel_time")